# Phase 3 -- Stream A: Strict-Unsupervised Anomaly Detection (IF + LOF)

**Thin research notebook.** All implementation lives in `src/mplad/stream_a/phase3.py`; this notebook imports it, explains the experiment, and controls execution. It deliberately does **not** duplicate the pipeline code.

**Historical sources** (preserved, read-only):
- `notebooks/archive/phase3_trackA_deprecated_2cell.ipynb` -- superseded contamination-based prototype (SHA-256 `D91C...A5E9DD`).
- `notebooks/archive/phase3_trackA_recovered_colab_14cell.ipynb` -- full research history; **cell index 4** is the revised primary 30k experiment extracted here. Cells 0-3 (Drive mount, contamination grids, earlier full-data native run) are deprecated; cells 5+ belong to Phases 4-6 and later experiments and are out of scope.

## Objective and experimental boundary

- Fit **5 Isolation Forest** and **5 LOF** configurations on one reproducible **30,000-row subsample** of FS-A (`ULB_FS_A_Baseline.csv`).
- **Strict unsupervised:** no `Class`, no `ground_truth_registry.csv`, no `contamination` parameter. Flags come from the post-hoc **"bottom 1% of native scores"** rule.
- **Score convention (native, preserved):** lower = more anomalous (IF `decision_function`, LOF `negative_outlier_factor_`). The `anomaly_score = -native` negation belongs to Phase 4/5, never here.
- The same 30,000 sampled rows (same order) feed every one of the ten runs.
- **Out of scope:** Phase 4 evaluation, Phase 5 diagnostics, Phase 6 agreement, and the larger novelty-mode experiment (to be specified separately after its artifacts are inspected).

Opening this notebook and running its cells top-to-bottom does **not** train anything: the execution cell is guarded by `RUN_EXPERIMENT = False`.

## 0 -- Bootstrap: import the project package

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upwards until a directory containing src/mplad is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "mplad").is_dir():
            return candidate
    raise FileNotFoundError(
        f"No repository root (directory containing 'src/mplad') found above {start}. "
        "Launch Jupyter from the repository root or a subdirectory of it."
    )


REPO_ROOT = find_repo_root(Path.cwd())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from mplad.common import config
from mplad.common.io import load_feature_matrix, load_predictions_pickle
from mplad.common.validation import validate_feature_matrix
from mplad.stream_a import phase3

print("repository root :", REPO_ROOT)
print("package         : mplad (imported from", config.__file__, ")")

## 1 -- Input: the frozen Phase 2 feature space (FS-A)

| Property | Value |
|---|---|
| File | `data/processed/stream-a/features/ULB_FS_A_Baseline.csv` |
| Rows | 284,807 (one per ULB transaction) |
| Columns | `row_id` + 30 features: `Time`, `V1`-`V28`, `Amount` |
| Scaling | `RobustScaler` on `Time` and `Amount`; `V1`-`V28` untouched (already PCA output) |
| Labels | **absent by construction** -- `Class` lives only in `ground_truth_registry.csv`, which this phase never reads |

`row_id` is retained for every observation (it is the join key used by Phase 4) but is removed from the model input. FS-B (`ULB_FS_B_LogAmount.csv`) is **not** used in this primary experiment.

In [ ]:
# Override here (str/Path) to point at a different copy of FS-A.
FS_A_PATH = None

fs_a_path = Path(FS_A_PATH) if FS_A_PATH is not None else config.DEFAULT_FS_A_PATH
print("FS-A path :", fs_a_path)
print("exists    :", fs_a_path.is_file())

fs_df = load_feature_matrix(fs_a_path)
feature_columns = validate_feature_matrix(fs_df)  # rejects 'Class', NaNs, duplicate row_ids, ...

print(f"rows      : {len(fs_df):,}")
print(f"columns   : {config.ID_COLUMN} + {len(feature_columns)} features "
      f"({feature_columns[0]} ... {feature_columns[-1]})")
print(f"row_id    : {int(fs_df[config.ID_COLUMN].min())} .. {int(fs_df[config.ID_COLUMN].max())}, "
      f"{fs_df[config.ID_COLUMN].nunique():,} unique")
print(f"NaN cells : {int(fs_df.isna().sum().sum())}")
print("schema validation: PASSED")

## 2 -- The reproducible 30,000-row sample

Cell index 4 of the recovered notebook draws the sample with the legacy NumPy API:

```python
np.random.seed(42)
sample_indices = np.random.choice(len(fs_df), size=30000, replace=False)
fs_subsampled = fs_df.iloc[sample_indices].reset_index(drop=True)
```

`phase3.build_experiment_sample` reproduces this draw with an isolated `np.random.RandomState(42)` -- the same legacy MT19937 stream as `np.random.seed(42)`, so the selected `row_id` sequence is identical (the test suite asserts bit-equality with the global API). The draw is label-blind: sampling happens before any model sees the data and never consults `Class`.

This cell only **draws** the sample (cheap, no fitting). The execution cell re-derives the same sample internally.

In [ ]:
SAMPLE_SIZE = config.SAMPLE_SIZE_DEFAULT   # 30,000
SAMPLE_SEED = config.SAMPLE_SEED_DEFAULT   # 42

row_ids, X = phase3.build_experiment_sample(fs_df, SAMPLE_SIZE, SAMPLE_SEED)

# Determinism self-check: a second, independent draw must be identical.
row_ids_2, X_2 = phase3.build_experiment_sample(fs_df, SAMPLE_SIZE, SAMPLE_SEED)
assert np.array_equal(row_ids, row_ids_2) and np.array_equal(X, X_2)

assert np.unique(row_ids).size == SAMPLE_SIZE

print(f"sample      : {X.shape[0]:,} rows x {X.shape[1]} features (dtype {X.dtype})")
print(f"row_id head : {row_ids[:5].tolist()} ... tail: {row_ids[-5:].tolist()}")
print("determinism : PASSED (two independent draws are identical)")

## 3 -- Model configuration overview

Ten runs, all on the same sample; **no `contamination` anywhere** (flags are post-hoc):

- **Isolation Forest -- 5 runs:** one baseline, one parameter-sensitivity run on `n_estimators`, one on `max_samples` (5% sub-sampling), and two seed-stability runs (`random_state` 43 and 44). Fixed: `n_jobs=1`.
- **LOF -- 5 runs:** neighbourhood-scale sweep (`n_neighbors` 5/15/30/50, euclidean) plus one metric-sensitivity run (manhattan). Fixed: `algorithm='ball_tree'`, `leaf_size=40`, `n_jobs=1`.
- **Flagging rule:** bottom 1% of native scores -> ~300 flags per run on a 30,000-row sample.

In [ ]:
print("Isolation Forest grid (+ fixed params:", config.IF_FIXED_PARAMS, ")")
for i, cfg in enumerate(config.ISOLATION_FOREST_CONFIGS, start=1):
    print(f"  IF_{i:03d}: {cfg}")

print("\nLOF grid (+ fixed params:", config.LOF_FIXED_PARAMS, ")")
for i, cfg in enumerate(config.LOF_CONFIGS, start=1):
    print(f"  LOF_{i:03d}: {cfg}")

print("\nflagging rule :", config.THRESHOLD_RULE_LABEL)
print("expected flags: ~", int(SAMPLE_SIZE * config.THRESHOLD_PERCENTILE / 100), "per run")
print("score storage : native orientation (lower = more anomalous), float32 scores / uint8 flags")

## 4 -- Execution (explicitly controlled)

Nothing runs unless `RUN_EXPERIMENT` is set to `True`. Expect the canonical run to take roughly **10-20 minutes**, dominated by the LOF grid (LOF_001 with `n_neighbors=5` is the slowest); Isolation Forest adds only seconds.

Outputs (overwrite-protected): `phase3_run_metadata.csv` and `phase3_canonical_predictions.pkl`. The canonical artifacts in `results/stream a/phase 3/` are not touched unless you point `OUTPUT_DIR` there **and** set `OVERWRITE = True` -- for a reproduction run, redirect `OUTPUT_DIR` instead.

In [ ]:
RUN_EXPERIMENT = False   # <<< set True to fit the ten models; nothing runs otherwise
OVERWRITE = False        # must be True to replace existing output files
OUTPUT_DIR = None        # None -> config.DEFAULT_OUTPUT_DIR; set a str/Path to redirect

if not RUN_EXPERIMENT:
    print("RUN_EXPERIMENT is False -- no model was fitted, nothing was written.")
    print("Set RUN_EXPERIMENT = True and re-run this cell to execute Phase 3.")
else:
    results = phase3.run_phase3_experiment(
        fs_a_path=fs_a_path,
        output_dir=Path(OUTPUT_DIR) if OUTPUT_DIR is not None else config.DEFAULT_OUTPUT_DIR,
        sample_size=SAMPLE_SIZE,
        sample_seed=SAMPLE_SEED,
        overwrite=OVERWRITE,
    )
    display(results["metadata"])

## 5 -- Artifact loading and verification

Reloads both artifacts from disk (independent of the in-memory `results`) and checks the output contract: bundle structure, run ordering, dtypes, lengths, flag counts against the metadata CSV, and native score orientation.

In [ ]:
ARTIFACT_DIR = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else config.DEFAULT_OUTPUT_DIR
meta_path = ARTIFACT_DIR / config.METADATA_FILENAME
pkl_path = ARTIFACT_DIR / config.PREDICTIONS_FILENAME

if not (meta_path.is_file() and pkl_path.is_file()):
    print("Artifacts not found -- run the execution cell (section 4) first.")
    print("  expected:", meta_path)
    print("  expected:", pkl_path)
else:
    metadata_df = pd.read_csv(meta_path)
    bundle = load_predictions_pickle(pkl_path)

    # --- bundle structure ------------------------------------------------
    assert set(bundle) == {"row_ids", "predictions"}
    run_ids = list(bundle["predictions"])
    assert run_ids == metadata_df["run_id"].tolist(), "run order mismatch vs metadata"

    n_rows = len(bundle["row_ids"])
    for run_id in run_ids:
        entry = bundle["predictions"][run_id]
        assert set(entry) == {"scores", "flags"}
        assert entry["scores"].dtype == np.float32
        assert entry["flags"].dtype == np.uint8
        assert entry["scores"].shape == entry["flags"].shape == (n_rows,)

        # flag counts agree with the metadata CSV
        flagged_in_csv = int(metadata_df.loc[metadata_df["run_id"] == run_id, "flagged_count"].iloc[0])
        assert int(entry["flags"].sum()) == flagged_in_csv

        # native orientation: flagged scores are the lowest tail
        flagged = entry["scores"][entry["flags"] == 1]
        unflagged = entry["scores"][entry["flags"] == 0]
        assert flagged.max() <= unflagged.min()

    # --- sample agreement ------------------------------------------------
    assert np.unique(bundle["row_ids"]).size == n_rows

    print(f"verified {len(run_ids)} runs x {n_rows:,} rows")
    print("bundle structure, dtypes, flag counts, score orientation: ALL OK")
    display(metadata_df)

## What this notebook deliberately does not do

- No Phase 4 label-based evaluation, Phase 5 diagnostics/plots, or Phase 6 agreement analysis -- separate stages.
- No larger "novelty mode" experiment -- it will be specified separately after its artifacts are inspected and its exact dataset size is confirmed.
- No refit on open: every execution is behind `RUN_EXPERIMENT`.
- No label access: `Class` / `ground_truth_registry.csv` are never read here.

Provenance: record any actual execution (paths, sample size/seed, runtimes) in `docs/provenance_log.md`.